> **Public-repository preparation copy.** This notebook is based on the corresponding final-run notebook used for the study. Execution outputs were cleared for publication. Set the `QA_DATASET_CSV` environment variable (or edit the default path below) to the location of `finetune_dataset_fixed_3000.csv` before running. Do not place access tokens in notebook source.


# Base stuff

In [ ]:
# Step 1: Install PyTorch and xFormers for CUDA 12.8
%pip install torch==2.10.0 torchvision==0.25.0 xformers==0.0.35 --index-url https://download.pytorch.org/whl/cu128

# Step 2: Install Unsloth and other dependencies separately
%pip install unsloth==2026.5.4 transformers==4.56.2 triton==3.6.0

In [ ]:
# Install LlamaIndex and dependencies for vector-based retrieval
! pip install llama-index==0.12.44 llama-index-vector-stores-chroma chromadb==1.0.13 \
    "opentelemetry-api==1.38.0" \
    "opentelemetry-sdk==1.38.0" \
    "opentelemetry-exporter-otlp-proto-grpc==1.38.0" \
    "opentelemetry-exporter-otlp-proto-common==1.38.0" \
    "opentelemetry-proto==1.38.0" \
    "opentelemetry-semantic-conventions==0.59b0" \
    "posthog<6.0.0"

# Install Hugging Face Hub utilities and LlamaIndex integrations
! pip install huggingface_hub==0.36.2 llama-index-llms-huggingface==0.5.0 llama-index-embeddings-huggingface --no-deps

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
 
import torch

from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core import Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
 
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
chromadb_collection_name = "collection_name"
chromadb_path = "/kaggle/working/VectorDbStorage"
 
db = chromadb.PersistentClient(path=chromadb_path)
 
collections = [c.name for c in db.list_collections()]
 
if chromadb_collection_name in collections:
    db.delete_collection(name=chromadb_collection_name)
 
chroma_collection = db.create_collection(name=chromadb_collection_name)

In [ ]:
# Load 4-bit quantized LLaMA 3.1 model using Unsloth for efficient inference
model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit"
 
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    load_in_4bit=True,
    device_map={'': torch.cuda.current_device()},
)
 
model = FastLanguageModel.for_inference(model)

In [ ]:
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

In [ ]:
llm = HuggingFaceLLM(
    model=model,
    tokenizer=tokenizer,
    context_window=2048,
    max_new_tokens=300,
    generate_kwargs={
        "do_sample": False,
    }
)

In [ ]:
# FIX 5: Use a stronger embedding model (bge-large instead of bge-base)
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-large-en-v1.5"
)

In [ ]:
# FIX 4: Disable chunking — each row is already a small, self-contained context.
# Splitting destroys cross-sentence relationships and hurts retrieval quality.
node_parser = SimpleNodeParser.from_defaults(
    chunk_size=2048,
    chunk_overlap=0,
)
 
Settings.llm = llm
Settings.embed_model = embed_model
Settings.node_parser = node_parser

# Data prepration

In [ ]:
import pandas as pd
from llama_index.core.schema import Document
 
import os
csv_path = os.environ.get('QA_DATASET_CSV', '/kaggle/input/datasets/lemika0/llamaindex-rag-deepseek-normalized-dataset/finetune_dataset_fixed_3000.csv')
df = pd.read_csv(csv_path)
 
df = df.dropna(subset=["context", "question"])

In [ ]:
# Use the pre-existing "split" column to divide the dataset.
# Expected values: "train", "validation", "test"
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "validation"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)
 
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
# FIX 1 + FIX 3: Index ONLY train + validation contexts (no test leakage).
# FIX 3: Store "Question + Context + Answer" in each Document so that
#         similarity is computed between the test question and
#         question+context pairs — much stronger than context-only matching.
#
# Each Document also carries the original fields as metadata so we can
# surface them later as few-shot examples in the prompt.
 
index_docs = []
 
for split_df in [train_df, val_df]:
    for _, row in split_df.iterrows():
        combined_text = (
            f"Question:\n{row['question']}\n\n"
            f"Context:\n{row['context']}"
        )
        doc = Document(
            text=combined_text,
            metadata={
                "question": row["question"],
                "context":  row["context"],
                "answer":   str(row.get("answer", "")),
            },
        )
        index_docs.append(doc)

In [ ]:
vector_store   = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
 
index = VectorStoreIndex.from_documents(
    documents=index_docs,
    storage_context=storage_context,
    show_progress=True,
)

# Data querying

In [ ]:
# Token budget constants — keep the total prompt well under 2048 tokens.
# A single context passage can be 300-500 tokens. With 3 few-shot examples
# that is already 900-1500 tokens before the test question even appears.
# These limits guarantee the test question + context always fits.
MAX_PROMPT_TOKENS   = 1800   # hard ceiling fed to the tokenizer
MAX_EXAMPLE_CHARS   = 400    # max chars kept per retrieved context in few-shot block
MAX_TEST_CTX_CHARS  = 3000   # max chars kept from the test context

SYSTEM_PROMPT = (
    "You are a helpful AI assistant."
    "When a user provides context, use only that context to answer their question."
    "If the question includes multiple sub-questions, answer each one concisely on a new line, in the same order."
    "Always strive to provide clear, structured, and accurate answers."
)

def truncate(text, max_chars):
    """Hard-truncate a string to max_chars characters."""
    return text if len(text) <= max_chars else text[:max_chars] + "..."

def build_prompt(test_context, test_question, retrieved_examples):
    """
    Build a few-shot retrieval-augmented prompt that fits within the
    token budget.  The key changes vs the previous version:

    1.  System prompt now explicitly demands short, extractive answers —
        this closes the style gap between the base model and the fine-tuned
        model which learned terse answers from the training data.

    2.  Retrieved contexts in the few-shot block are hard-truncated so they
        don't dominate the token budget.

    3.  The test context is also truncated as a safety valve.

    4.  We progressively drop few-shot examples if the tokenised prompt is
        still too long, falling back to zero-shot if necessary.
    """

    def assemble(examples):
        examples_block = ""
        for i, ex in enumerate(examples, 1):
            short_ctx = truncate(ex["context"], MAX_EXAMPLE_CHARS)
            examples_block += (
                f"\n--- Example {i} ---\n"
                f"Context: {short_ctx}\n"
                f"Question: {ex['question']}\n"
                f"Answer: {ex['answer']}\n"
            )

        short_test_ctx = truncate(test_context, MAX_TEST_CTX_CHARS)

        user_content = ""
        if examples_block:
            user_content += (
                "Here are a few solved examples showing the expected answer style:\n"
                + examples_block
                + "\nNow answer the question below using ONLY the context provided.\n\n"
            )

        user_content += (
            f"Context:\n{short_test_ctx}\n\n"
            f"Question: {test_question}\n"
            f"Answer:"
        )

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_content},
        ]

        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    # Try with all examples; drop one at a time if over budget
    candidates = list(retrieved_examples)
    while True:
        prompt = assemble(candidates)
        token_len = len(tokenizer.encode(prompt))
        if token_len <= MAX_PROMPT_TOKENS or len(candidates) == 0:
            break
        candidates.pop()   # drop the last (least relevant) example

    return prompt

In [ ]:
import torch
import time
import csv
import os

# ---------------------------------------------------------
# GPU memory tracking configuration
# ---------------------------------------------------------
assert torch.cuda.is_available(), "CUDA GPU is not available."

device = torch.cuda.current_device()
gpu_name = torch.cuda.get_device_name(device)

# Reset peak memory statistics before the measured run
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

start_time = time.time()

print(f"GPU: {gpu_name}")
print(f"GPU index: {device}")
print("Starting QA inference loop...\n")

In [ ]:
# FIX 6: Lower top_k to reduce noise. top_k=3 is a good starting point.
retriever = index.as_retriever(similarity_top_k=3)

In [ ]:
# ── helper: strip DeepSeek-R1 chain-of-thought tags ───────────────────────────
import re

def strip_think_tags(text: str) -> str:
    """
    DeepSeek-R1 reasoning models wrap their internal chain-of-thought in
    <think>...</think> blocks. Remove the entire block and return only the
    final answer that follows.

    Handles three cases:
      1. Full block present:  '<think>...\n</think>\n\nAnswer'  → 'Answer'
      2. Only close tag left: '</think>\n\nAnswer'              → 'Answer'
         (happens when the open tag was part of the prompt slice boundary)
      3. No tags at all:      'Answer'                          → 'Answer'  (no-op)
    """
    # Remove a full <think>...</think> block (non-greedy, DOTALL)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    # Remove any stray closing tag
    text = re.sub(r"</think>", "", text)
    # Remove any stray opening tag
    text = re.sub(r"<think>", "", text)
    return text.strip()
# ──────────────────────────────────────────────────────────────────────────────

answers  = []
contexts = []
n_truncated = 0   # diagnostic: how many prompts hit zero-shot fallback

with torch.inference_mode():

    for idx, row in test_df.iterrows():

        test_question = row["question"]
        test_context  = row["context"]

        nodes = retriever.retrieve(test_question)

        retrieved_examples = []
        for node in nodes:
            meta = node.metadata
            if meta.get("answer"):
                retrieved_examples.append({
                    "question": meta["question"],
                    "context":  meta["context"],
                    "answer":   meta["answer"],
                })

        prompt = build_prompt(test_context, test_question, retrieved_examples)

        token_len = len(tokenizer.encode(prompt))
        if token_len > MAX_PROMPT_TOKENS:
            n_truncated += 1

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_PROMPT_TOKENS,
        ).to("cuda")

        input_length = inputs["input_ids"].shape[1]

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
        )

        raw_answer = tokenizer.decode(
            outputs[0][input_length:],
            skip_special_tokens=True,
        ).strip()

        # ── NEW: remove DeepSeek-R1 thinking block ──
        answer = strip_think_tags(raw_answer)
        # ────────────────────────────────────────────

        answers.append(answer)
        contexts.append(test_context)

print(f"Done. Prompts that exceeded budget (zero-shot fallback): {n_truncated}/{len(test_df)}")

In [ ]:
# Synchronize so all CUDA operations finish before measuring
torch.cuda.synchronize()

elapsed_time = time.time() - start_time

# Peak memory statistics
peak_allocated = torch.cuda.max_memory_allocated(device)
peak_reserved = torch.cuda.max_memory_reserved(device)

# Current memory statistics
current_allocated = torch.cuda.memory_allocated(device)
current_reserved = torch.cuda.memory_reserved(device)

def gb(x):
    return x / (1024 ** 3)

print("\n" + "=" * 60)
print("GPU MEMORY REPORT")
print("=" * 60)
print(f"GPU:                    {gpu_name}")
print(f"QA inference-loop time:          {elapsed_time / 3600:.2f} h")
print(f"Peak allocated memory:  {gb(peak_allocated):.3f} GB")
print(f"Peak reserved memory:   {gb(peak_reserved):.3f} GB")
print(f"Final allocated memory: {gb(current_allocated):.3f} GB")
print(f"Final reserved memory:  {gb(current_reserved):.3f} GB")
print("=" * 60)

In [ ]:
# ---------------------------------------------------------
# Save result to CSV
# ---------------------------------------------------------
log_file = "gpu_memory_log.csv"

file_exists = os.path.exists(log_file)

with open(log_file, "a", newline="") as f:
    writer = csv.writer(f)

    if not file_exists:
        writer.writerow([
            "gpu",
            "qa_inference_loop_sec",
            "peak_allocated_gb",
            "peak_reserved_gb",
            "final_allocated_gb",
            "final_reserved_gb"
        ])

    writer.writerow([
        gpu_name,
        round(elapsed_time, 2),
        round(gb(peak_allocated), 3),
        round(gb(peak_reserved), 3),
        round(gb(current_allocated), 3),
        round(gb(current_reserved), 3)
    ])

print(f"\nSaved to: {log_file}")

In [ ]:
# Save results — only test split rows with their predicted answers
test_df = test_df.copy()
test_df["predicted_answer"]    = answers
test_df["retrieved_context"]   = contexts
 
test_df.to_csv('/kaggle/working/sampled_dataset_test_split_only.csv', index=False)